In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# =========================================================
# 10 JUDUL SKRIPSI
# =========================================================

dokumen = [
    "Analisis Pola Cuaca Harian Kota Makassar Menggunakan Metode Klasterisasi Hibrida DBSCAN-Gaussian Mixture Model",

    "Analisis Kerentanan Website E-Skripsi JTIK dan Rancangan Solusi Keamanan Berdasarkan OWASP WSTG",

    "Pengaruh Literasi Digital terhadap Optimalisasi Pemanfaatan Artificial Intelligence (AI) pada Pembelajaran Mandiri Mahasiswa Pendidikan Teknik Informatika dan Komputer FT UNM",

    "Pengaruh Self-Determination terhadap Disengagement Academic Mahasiswa PTIK UNM",

    "Pengembangan Chatbot Akademik Berbasis Telegram: Integrasi Retrieval Augmented Generation (RAG) dengan Model LLM untuk Layanan Informasi Mahasiswa JTIK FT UNM",

    "Pengembangan Modul Coding for Kids Thunkable dalam Belajar Membuat Game dengan Code Block di Algoland Academy",

    "Pengaruh Pemanfaatan AI Generatif terhadap Efisiensi Waktu dan Produktivitas Penulisan Berita Real-Time di Stasiun Berita Tribun Timur",

    "Evaluasi Augmentasi Sinonim dengan SBERT dalam Menangani Variasi Leksikal pada Klasifikasi Teks Milenial dan Gen Z",

    "Analisis Label Nutrisi Minuman Kemasan Secara Terintegrasi Menggunakan YOLOv8, Optical Character Recognition, dan Support Vector Machine untuk Penilaian Risiko Kesehatan",

    "Sistem Klasifikasi Aktivitas Mahasiswa Tingkat Akhir Jurusan Teknik Informatika dan Komputer FT UNM dengan Metode Support Vector Machine"
]


# =========================================================
# QUERY DAN GROUND TRUTH
# =========================================================

queries = {
    "artificial intelligence ai generatif chatbot llm": {3, 5, 7},
    "jaringan komputer telegram website": set(),
    "iot sistem teknologi perangkat ": set()
}


# =========================================================
# TF-IDF
# =========================================================

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(dokumen)


# =========================================================
# FUNGSI EVALUASI
# =========================================================

def precision_at_k(hasil, relevan, k):
    if k == 0:
        return 0.0

    cocok = sum(1 for d in hasil[:k] if d in relevan)
    return cocok / k


def recall(hasil, relevan):
    if len(relevan) == 0:
        return 0.0

    cocok = sum(1 for d in hasil if d in relevan)
    return cocok / len(relevan)


def f1(p, r):
    if p + r == 0:
        return 0.0

    return 2 * p * r / (p + r)


def average_precision(hasil, relevan):
    if len(relevan) == 0:
        return 0.0

    hit = 0
    total = 0.0

    for i, d in enumerate(hasil, start=1):

        if d in relevan:
            hit += 1
            total += hit / i

    return total / len(relevan)


# =========================================================
# PROSES MESIN PENCARI
# =========================================================

nilai_ap = []

for query, relevan in queries.items():

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    # Mengubah query menjadi TF-IDF
    query_vector = vectorizer.transform([query])

    # Menghitung cosine similarity
    similarity = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Mengurutkan dokumen berdasarkan skor tertinggi
    ranking = similarity.argsort()[::-1]

    # Mengubah index menjadi nomor dokumen 1-10
    hasil = [index + 1 for index in ranking]

    # =====================================================
    # 3 DOKUMEN TERATAS
    # =====================================================

    print("\n--- 3 DOKUMEN TERATAS ---")

    for peringkat, index in enumerate(ranking[:3], start=1):

        print("\nPeringkat", peringkat)
        print("ID Dokumen :", index + 1)
        print("Judul      :", dokumen[index])
        print("Skor       :", f"{similarity[index]:.4f}")

    # =====================================================
    # EVALUASI
    # =====================================================

    p = precision_at_k(hasil, relevan, 3)
    r = recall(hasil, relevan)
    f = f1(p, r)
    ap = average_precision(hasil, relevan)

    nilai_ap.append(ap)

    print("\n--- GROUND TRUTH ---")
    print("Dokumen relevan :", relevan)

    print("\n--- HASIL EVALUASI ---")
    print("Precision@3      :", f"{p:.4f}")
    print("Recall           :", f"{r:.4f}")
    print("F1-Score         :", f"{f:.4f}")
    print("Average Precision:", f"{ap:.4f}")


# =========================================================
# MAP
# =========================================================

MAP = sum(nilai_ap) / len(nilai_ap)

print("\n" + "=" * 80)
print("MEAN AVERAGE PRECISION (MAP)")
print("=" * 80)

print("MAP :", f"{MAP:.4f}")